In [1]:
import json
import pandas as pd 

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from preprocessing.loader import load_phiusiil
from preprocessing.validator import (
    validate_dataframe,
    print_dataset_summary
)
from preprocessing.splitter import split_dataset

from preprocessing.duplicate_detector import (
    find_duplicate_columns,
    remove_duplicate_columns
)
from preprocessing.config import ARTIFACTS_DIR 
from preprocessing.outlier_handler import(
    compute_outlier_cap,
    apply_outlier_caps
)
from preprocessing.column_handler import drop_unused_columns
from utils.io_utils import save_json
from preprocessing.variance_filter import(
    compute_variance_filter,
    apply_variance_filter
)
from preprocessing.correlation_filter import(
    compute_correlation_filter,
    apply_correlation_filter
)
from preprocessing.mutual_information import compute_mutual_information
from preprocessing.feature_selector import select_features
from preprocessing.artifact_builder import build_feature_selection_artifact

In [2]:
phiusiil_df = load_phiusiil()

validate_dataframe(phiusiil_df)

print_dataset_summary(phiusiil_df)

Shape : (235795, 56)
<class 'pandas.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235795 non-null  str    
 1   URL                         235795 non-null  str    
 2   URLLength                   235795 non-null  int64  
 3   Domain                      235795 non-null  str    
 4   DomainLength                235795 non-null  int64  
 5   IsDomainIP                  235795 non-null  int64  
 6   TLD                         235795 non-null  str    
 7   URLSimilarityIndex          235795 non-null  float64
 8   CharContinuationRate        235795 non-null  float64
 9   TLDLegitimateProb           235795 non-null  float64
 10  URLCharProb                 235795 non-null  float64
 11  TLDLength                   235795 non-null  int64  
 12  NoOfSubDomain               235795 non-null  int64  
 13  HasO

In [3]:
print("=" * 60)
print("Raw Dataset")
print("=" * 60)

print(f"Shape : {phiusiil_df.shape}")
print(f"Duplicate rows : {phiusiil_df.duplicated().sum()}")

Raw Dataset
Shape : (235795, 56)
Duplicate rows : 0


In [4]:
url_counts = phiusiil_df["URL"].value_counts()

duplicate_urls = url_counts[url_counts > 1]

print("=" * 60)
print("Raw Dataset URL Duplicate Audit")
print("=" * 60)

print(f"Unique URLs             : {phiusiil_df['URL'].nunique()}")
print(f"Duplicate URL values    : {len(duplicate_urls)}")
print(f"Rows belonging to duplicates : {duplicate_urls.sum()}")

Raw Dataset URL Duplicate Audit
Unique URLs             : 235370
Duplicate URL values    : 425
Rows belonging to duplicates : 850


In [5]:
url_label_counts = (
    phiusiil_df.groupby("URL")["label"]
    .nunique()
)

conflicting_urls = url_label_counts[
    url_label_counts > 1
]

print("=" * 60)
print("URL Label Conflict Audit")
print("=" * 60)

print(f"URLs with conflicting labels : {len(conflicting_urls)}")

URL Label Conflict Audit
URLs with conflicting labels : 0


In [6]:
df_clean = (
    phiusiil_df
    .drop_duplicates(subset="URL", keep="first")
    .reset_index(drop=True)
)

In [7]:
print("=" * 60)
print("After URL Deduplication")
print("=" * 60)

print(f"Shape : {df_clean.shape}")
print(f"Unique URLs : {df_clean['URL'].nunique()}")
print(f"Duplicate URLs : {df_clean['URL'].duplicated().sum()}")

After URL Deduplication
Shape : (235370, 56)
Unique URLs : 235370
Duplicate URLs : 0


In [8]:
print(df_clean["Domain"].nunique())
print(df_clean["Domain"].isna().sum())

220086
0


In [9]:
print(
    df_clean["Domain"]
    .value_counts()
    .head(10)
)

Domain
ipfs.io                         1192
docs.google.com                  526
mail.deliverylifesupport.com     370
cloudflare-ipfs.com              359
storageapi.fleek.co              318
gateway.pinata.cloud             275
gateway.ipfs.io                  249
ipfs.litnet.work                 223
cf-ipfs.com                      190
s3.amazonaws.com                 182
Name: count, dtype: int64


In [10]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)


train_idx, temp_idx = next(
    gss.split(
        df_clean,
        y=df_clean["label"],
        groups=df_clean["Domain"]
    )
)


train_df = df_clean.iloc[train_idx].copy()
temp_df = df_clean.iloc[temp_idx].copy()

In [11]:
gss_temp = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss_temp.split(
        temp_df,
        y=temp_df["label"],
        groups=temp_df["Domain"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

In [12]:
train_domains = set(train_df["Domain"])
val_domains = set(val_df["Domain"])
test_domains = set(test_df["Domain"])

print("=" * 60)
print("Domain Split Verification")
print("=" * 60)

print(f"Train domains      : {len(train_domains)}")
print(f"Validation domains : {len(val_domains)}")
print(f"Test domains       : {len(test_domains)}")

print(f"Train ∩ Val        : {len(train_domains & val_domains)}")
print(f"Train ∩ Test       : {len(train_domains & test_domains)}")
print(f"Val ∩ Test         : {len(val_domains & test_domains)}")

Domain Split Verification
Train domains      : 176068
Validation domains : 22009
Test domains       : 22009
Train ∩ Val        : 0
Train ∩ Test       : 0
Val ∩ Test         : 0


In [13]:
print("=" * 60)
print("Class Distribution")
print("=" * 60)

print("\nTrain:")
print(train_df["label"].value_counts(normalize=True))

print("\nValidation:")
print(val_df["label"].value_counts(normalize=True))

print("\nTest:")
print(test_df["label"].value_counts(normalize=True))

Class Distribution

Train:
label
1    0.571835
0    0.428165
Name: proportion, dtype: float64

Validation:
label
1    0.589307
0    0.410693
Name: proportion, dtype: float64

Test:
label
1    0.565709
0    0.434291
Name: proportion, dtype: float64


In [14]:
train_urls = set(train_df["URL"])
val_urls = set(val_df["URL"])
test_urls = set(test_df["URL"])

print("=" * 60)
print("Cross-Split URL Verification")
print("=" * 60)

print(f"Train URLs      : {len(train_urls)}")
print(f"Validation URLs : {len(val_urls)}")
print(f"Test URLs       : {len(test_urls)}")

print(f"Train ∩ Val     : {len(train_urls & val_urls)}")
print(f"Train ∩ Test    : {len(train_urls & test_urls)}")
print(f"Val ∩ Test      : {len(val_urls & test_urls)}")

Cross-Split URL Verification
Train URLs      : 188738
Validation URLs : 23005
Test URLs       : 23627
Train ∩ Val     : 0
Train ∩ Test    : 0
Val ∩ Test      : 0


In [15]:
print("=" * 60)
print("Split Sizes")
print("=" * 60)

print(f"Train      : {train_df.shape}")
print(f"Validation : {val_df.shape}")
print(f"Test       : {test_df.shape}")

Split Sizes
Train      : (188738, 56)
Validation : (23005, 56)
Test       : (23627, 56)


In [16]:
y_train = train_df["label"].copy()
y_val = val_df["label"].copy()
y_test = test_df["label"].copy()

X_train = train_df.drop(columns=["label"]).copy()
X_val = val_df.drop(columns=["label"]).copy()
X_test = test_df.drop(columns=["label"]).copy()

In [17]:
print("=" * 60)
print("Fresh Split")
print("=" * 60)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

print("X_val   :", X_val.shape)
print("y_val   :", y_val.shape)

print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)

Fresh Split
X_train : (188738, 55)
y_train : (188738,)
X_val   : (23005, 55)
y_val   : (23005,)
X_test  : (23627, 55)
y_test  : (23627,)


In [18]:
train_urls = X_train["URL"].copy()
val_urls = X_val["URL"].copy()
test_urls = X_test["URL"].copy()

train_domains = X_train["Domain"].copy()
val_domains = X_val["Domain"].copy()
test_domains = X_test["Domain"].copy()

In [19]:
# Drop the unused columns 

X_train = drop_unused_columns(X_train)
X_val = drop_unused_columns(X_val)
X_test = drop_unused_columns(X_test)

In [20]:
duplicate_columns = find_duplicate_columns(X_train)

print(f"Duplicate Columns Found : {len(duplicate_columns)}")

duplicate_columns

Duplicate Columns Found : 0


{}

In [21]:
X_train = remove_duplicate_columns(X_train, duplicate_columns)
X_val   = remove_duplicate_columns(X_val, duplicate_columns)
X_test  = remove_duplicate_columns(X_test, duplicate_columns)

In [22]:
# Save duplicate_columns

with open(ARTIFACTS_DIR / "duplicate_columns.json", "w") as file:

    json.dump(
        duplicate_columns,
        file,
        indent=4
    )

In [23]:
X_train_before_cap = X_train.copy()
X_val_before_cap = X_val.copy()
print("Training duplicates :", X_train_before_cap.duplicated().sum())
print("Validation duplicates :", X_val_before_cap.duplicated().sum())

Training duplicates : 592
Validation duplicates : 10


In [24]:
# Checking caps of each column 
caps = compute_outlier_cap(X_train)

save_json(
    caps,
    ARTIFACTS_DIR / "outiler_caps.json"
)

In [25]:
print(len(caps))

50


In [26]:
X_train = apply_outlier_caps(X_train, caps)
X_val = apply_outlier_caps(X_val, caps)
X_test = apply_outlier_caps(X_test, caps)

In [27]:
print("=" * 60)
print("Outlier Handling Completed")
print("=" * 60)

print("Train Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

Outlier Handling Completed
Train Shape : (188738, 50)
Validation Shape : (23005, 50)
Test Shape : (23627, 50)


In [28]:
X_train_after_cap = X_train.copy()
X_val_after_cap = X_val.copy()
print("Training duplicates :", X_train_after_cap.duplicated().sum())
print("Validation duplicates :", X_val_after_cap.duplicated().sum())

Training duplicates : 1761
Validation duplicates : 86


In [29]:
X_train["IsHTTPS"].value_counts()

IsHTTPS
1    188738
Name: count, dtype: int64

In [30]:
X_train["HasPasswordField"].value_counts()

HasPasswordField
0    188738
Name: count, dtype: int64

In [31]:
X_train["Bank"].value_counts()

Bank
0    188738
Name: count, dtype: int64

In [32]:
X_train["NoOfSubDomain"].value_counts()

NoOfSubDomain
1    188738
Name: count, dtype: int64

In [33]:
# Variance Filter 

selected_features, removed_features = compute_variance_filter(
    X_train,
    threshold=0.0
)

In [34]:
save_json(
    {
        "selected_features": selected_features,
        "removed_features": removed_features
    },
    ARTIFACTS_DIR / "variance_filter.json"
)

In [35]:
X_train = apply_variance_filter(
    X_train,
    selected_features
)

X_val = apply_variance_filter(
    X_val,
    selected_features
)

X_test = apply_variance_filter(
    X_test,
    selected_features
)

In [36]:
print("=" * 60)
print("Variance Filter")
print("=" * 60)

print(f"Removed Features : {len(removed_features)}")
print(f"Remaining Features : {len(selected_features)}")

print("\nRemoved Features:")
print(removed_features)

Variance Filter
Removed Features : 20
Remaining Features : 30

Removed Features:
['IsDomainIP', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'IsHTTPS', 'HasTitle', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'NoOfPopup', 'HasExternalFormSubmit', 'HasPasswordField', 'Bank', 'Pay', 'Crypto']


In [37]:
# Mutual Information 

mi_scores, ranking = compute_mutual_information(
    X_train,
    y_train
)

In [38]:
save_json(
    {
        "ranking": ranking
    },
    ARTIFACTS_DIR / "mutual_information_scores.json"
)

In [39]:
print("=" * 60)
print("Top 10 Mutual Information Features")
print("=" * 60)

pd.DataFrame(ranking).head(10)

Top 10 Mutual Information Features


,rank,feature,mi_score
0,1,URLSimilarityIndex,0.676957
1,2,LineOfCode,0.600110
2,3,NoOfExternalRef,0.561515
3,4,NoOfImage,0.543614
4,5,NoOfSelfRef,0.527682
5,6,NoOfJS,0.498119
6,7,LargestLineLength,0.461771
7,8,NoOfCSS,0.447168
8,9,HasSocialNet,0.410364
9,10,LetterRatioInURL,0.383070


In [40]:
# Correlation filter 

selected_features, removed_features, decision_report = compute_correlation_filter(
    X_train,
    y_train,
    threshold=0.95
)

In [41]:
save_json(
    {
        "threshold": 0.95,
        "selected_features": selected_features,
        "removed_features": removed_features,
        "decision_report": decision_report
    },
    ARTIFACTS_DIR / "correlation_filter.json"
)

In [42]:
X_train = apply_correlation_filter(
    X_train,
    selected_features
)

X_val = apply_correlation_filter(
    X_val,
    selected_features
)

X_test = apply_correlation_filter(
    X_test,
    selected_features
)

In [43]:
print("=" * 60)
print("Correlation Filter")
print("=" * 60)

print(f"Removed Features : {len(removed_features)}")
print(f"Remaining Features : {len(selected_features)}")

print("\nRemoved Features")
for feature in removed_features:
    print(feature)

Correlation Filter
Removed Features : 2
Remaining Features : 28

Removed Features
URLTitleMatchScore
NoOfLettersInURL


In [44]:
# Feature Selection 

model, selector, selected_features, removed_features, ranking, threshold_value = select_features(
    X_train,
    y_train
)

In [45]:
X_train_processed = X_train[selected_features]

X_val_processed = X_val[selected_features]

X_test_processed = X_test[selected_features]

In [46]:
# Build the artifact for feature selection 

feature_selection_artifact = build_feature_selection_artifact(
    ranking=ranking,
    selected_features=selected_features,
    removed_features=removed_features,
    threshold_type="median",
    threshold_value=threshold_value,
    random_state=42
)

In [47]:
save_json(
    feature_selection_artifact,
    ARTIFACTS_DIR / "feature_selection.json"
)

In [48]:
# Verifying...

print("=" * 60)
print("Feature Selection")
print("=" * 60)

print(f"Selected Features : {len(selected_features)}")
print(f"Removed Features : {len(removed_features)}")

print("\nSelected Features:")
print(selected_features)

Feature Selection
Selected Features : 14
Removed Features : 14

Selected Features:
['URLSimilarityIndex', 'CharContinuationRate', 'LineOfCode', 'HasFavicon', 'Robots', 'HasDescription', 'HasSocialNet', 'HasHiddenFields', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfExternalRef']


In [49]:
print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

(188738, 14)
(23005, 14)
(23627, 14)


In [50]:
# ===========================
# Save Processed Features
# ===========================

X_train_processed.to_csv(
    ARTIFACTS_DIR / "X_train_processed.csv",
    index=False
)

X_val_processed.to_csv(
    ARTIFACTS_DIR / "X_val_processed.csv",
    index=False
)

X_test_processed.to_csv(
    ARTIFACTS_DIR / "X_test_processed.csv",
    index=False
)

# ===========================
# Save Labels
# ===========================

y_train.to_frame().to_csv(
    ARTIFACTS_DIR / "y_train.csv",
    index=False
)

y_val.to_frame().to_csv(
    ARTIFACTS_DIR / "y_val.csv",
    index=False
)

y_test.to_frame().to_csv(
    ARTIFACTS_DIR / "y_test.csv",
    index=False
)

In [51]:
print("=" * 60)
print("Within-Split Duplicate Check")
print("=" * 60)

print(f"Training duplicates   : {X_train_processed.duplicated().sum()}")
print(f"Validation duplicates : {X_val_processed.duplicated().sum()}")
print(f"Testing duplicates    : {X_test_processed.duplicated().sum()}")

Within-Split Duplicate Check
Training duplicates   : 23097
Validation duplicates : 1405
Testing duplicates    : 1628


In [52]:
train_features = set(map(tuple, X_train_processed.to_numpy()))
val_features = set(map(tuple, X_val_processed.to_numpy()))
test_features = set(map(tuple, X_test_processed.to_numpy()))

print("=" * 60)
print("Cross-Split Feature Duplicate Check")
print("=" * 60)

print(f"Train ∩ Validation : {len(train_features & val_features)}")
print(f"Train ∩ Test       : {len(train_features & test_features)}")
print(f"Validation ∩ Test  : {len(val_features & test_features)}")

Cross-Split Feature Duplicate Check
Train ∩ Validation : 2056
Train ∩ Test       : 1988
Validation ∩ Test  : 727


In [53]:
print("=" * 60)
print("Final Training Data")
print("=" * 60)

print("X_train :", X_train_processed.shape)
print("y_train :", y_train.shape)

print("X_val   :", X_val_processed.shape)
print("y_val   :", y_val.shape)

print("X_test  :", X_test_processed.shape)
print("y_test  :", y_test.shape)

Final Training Data
X_train : (188738, 14)
y_train : (188738,)
X_val   : (23005, 14)
y_val   : (23005,)
X_test  : (23627, 14)
y_test  : (23627,)


In [54]:
# Running cross split label analysis

train_check = X_train_processed.copy()
train_check["label"] = y_train.to_numpy()

val_check = X_val_processed.copy()
val_check["label"] = y_val.to_numpy()

test_check = X_test_processed.copy()
test_check["label"] = y_test.to_numpy()

In [55]:
feature_columns = X_train_processed.columns.tolist()

In [56]:
train_val_shared = train_check.merge(
    val_check,
    how="inner",
    on=feature_columns,
    suffixes=("_train", "_val")
)

print("=" * 60)
print("Train-Validation Feature Duplicate Label Analysis")
print("=" * 60)

print(f"Shared duplicate pairs : {len(train_val_shared)}")

print("\nLabel combinations:")
print(
    train_val_shared[
        ["label_train", "label_val"]
    ].value_counts()
)

Train-Validation Feature Duplicate Label Analysis
Shared duplicate pairs : 103674

Label combinations:
label_train  label_val
0            0            101958
1            1              1716
Name: count, dtype: int64


In [57]:
train_test_shared = train_check.merge(
    test_check,
    how="inner",
    on=feature_columns,
    suffixes=("_train", "_test")
)

print("=" * 60)
print("Train-Test Feature Duplicate Label Analysis")
print("=" * 60)

print(f"Shared duplicate pairs : {len(train_test_shared)}")

print("\nLabel combinations:")
print(
    train_test_shared[
        ["label_train", "label_test"]
    ].value_counts()
)

Train-Test Feature Duplicate Label Analysis
Shared duplicate pairs : 96597

Label combinations:
label_train  label_test
0            0             95009
1            1              1588
Name: count, dtype: int64


In [58]:
val_test_shared = val_check.merge(
    test_check,
    how="inner",
    on=feature_columns,
    suffixes=("_val", "_test")
)

print("=" * 60)
print("Validation-Test Feature Duplicate Label Analysis")
print("=" * 60)

print(f"Shared duplicate pairs : {len(val_test_shared)}")

print("\nLabel combinations:")
print(
    val_test_shared[
        ["label_val", "label_test"]
    ].value_counts()
)

Validation-Test Feature Duplicate Label Analysis
Shared duplicate pairs : 11472

Label combinations:
label_val  label_test
0          0             11237
1          1               235
Name: count, dtype: int64


In [59]:
train_features = set(
    map(tuple, X_train_processed.to_numpy())
)

val_features = set(
    map(tuple, X_val_processed.to_numpy())
)

test_features = set(
    map(tuple, X_test_processed.to_numpy())
)


train_val_shared = train_features & val_features
train_test_shared = train_features & test_features
val_test_shared = val_features & test_features


print("=" * 60)
print("Feature Representation Overlap")
print("=" * 60)

print(
    f"Validation rows with train representation : "
    f"{X_val_processed.apply(tuple, axis=1).isin(train_features).sum()}"
)

print(
    f"Test rows with train representation       : "
    f"{X_test_processed.apply(tuple, axis=1).isin(train_features).sum()}"
)

print(
    f"Test rows with validation representation  : "
    f"{X_test_processed.apply(tuple, axis=1).isin(val_features).sum()}"
)

Feature Representation Overlap
Validation rows with train representation : 3065
Test rows with train representation       : 2992
Test rows with validation representation  : 1516


In [60]:
val_overlap_count = (
    X_val_processed
    .apply(tuple, axis=1)
    .isin(train_features)
    .sum()
)

test_overlap_count = (
    X_test_processed
    .apply(tuple, axis=1)
    .isin(train_features)
    .sum()
)


print("\nOverlap percentages:")

print(
    f"Validation: "
    f"{val_overlap_count / len(X_val_processed) * 100:.2f}%"
)

print(
    f"Test: "
    f"{test_overlap_count / len(X_test_processed) * 100:.2f}%"
)


Overlap percentages:
Validation: 13.32%
Test: 12.66%


In [61]:
print("=" * 60)
print("Feature Representation Uniqueness")
print("=" * 60)

print(
    f"Train unique representations : "
    f"{len(train_features):,}"
)

print(
    f"Validation unique representations : "
    f"{len(val_features):,}"
)

print(
    f"Test unique representations : "
    f"{len(test_features):,}"
)

Feature Representation Uniqueness
Train unique representations : 165,641
Validation unique representations : 21,600
Test unique representations : 21,999


In [62]:
print("\nUniqueness percentage:")

print(
    f"Train      : "
    f"{len(train_features) / len(X_train_processed) * 100:.2f}%"
)

print(
    f"Validation : "
    f"{len(val_features) / len(X_val_processed) * 100:.2f}%"
)

print(
    f"Test       : "
    f"{len(test_features) / len(X_test_processed) * 100:.2f}%"
)


Uniqueness percentage:
Train      : 87.76%
Validation : 93.89%
Test       : 93.11%
